# Experimento 2 — Convergencia desde cero: Learned vs. RoPE

**Práctica 1 · Procesamiento del Lenguaje Natural I · Rama `development`**

Este cuaderno entrena y compara las **clases reales del equipo**, definidas en `minigpt/config.py`, `minigpt/model.py`, `minigpt/attention.py` y `minigpt/rope.py`.

- **Learned** (`positional="learned"`): GPT causal con embeddings posicionales absolutos entrenables (mecanismo del GPT-2 original).
- **RoPE** (`positional="rope"`): exactamente el mismo modelo salvo la codificación posicional por rotaciones de Q y K.

Los dos se crean **desde cero**, sin llamar a `pretrained_gpt2`, sin cargar pesos externos ni realizar *fine-tuning*.

**Hipótesis a contrastar (no afirmación de resultados):** RoPE podría disminuir antes la pérdida de validación. La invariancia relativa de RoPE no garantiza, por sí sola, una convergencia más rápida.

**Archivos generados:** curvas PNG, CSV de pérdidas y resumen, JSON de configuración y (opcionalmente) *checkpoints* del mejor modelo de cada variante.


## 1. Diseño del experimento y comparabilidad

| Aspecto | Control aplicado |
|---|---|
| **Código real** | Se importan `GPTConfig` y `GPTModel` de vuestro paquete `minigpt`, no se crea una clase alternativa. |
| **Tarea** | Predicción autoregresiva del siguiente carácter, con *cross-entropy*. |
| **Datos** | `data/corpus.txt`, o el archivo que indiquéis en `ARCHIVO_CORPUS`. |
| **Particiones** | Entrenamiento 90 % y validación 10 %: cortes contiguos **antes** de construir las ventanas. |
| **Tokenización** | Por caracteres. Vocabulario construido solo con el entrenamiento y un identificador `<UNK>` para caracteres de validación no vistos. |
| **Arquitectura** | Misma cantidad de capas, cabezas, longitud, dimensión, dropout y proyecciones. |
| **Pesos iniciales** | Reinicio de semilla antes de crear cada variante y prueba de igualdad exacta en los tensores comunes. |
| **Lotes** | DataLoader nuevo con igual semilla, datos y orden para ambas variantes. |
| **Optimización** | AdamW, tasa de aprendizaje, weight decay, grad clipping y número de épocas idénticos. |
| **Métricas** | Pérdida de entrenamiento por lote y por época; pérdida de validación antes de entrenar y después de cada época. |

**Matices:** Learned incorpora una tabla posicional adicional con `context_length × emb_dim` parámetros. Mantener esta diferencia es correcto: es justamente parte de las dos técnicas que se comparan. Sin embargo, no significa que los modelos tengan exactamente el mismo número de parámetros.

El uso de caracteres simplifica el cálculo para entrenar en minutos; **no reproduce la tokenización BPE del GPT-2 de OpenAI**, aunque sí emplea vuestra implementación `GPTModel`.


In [ ]:
from pathlib import Path
import os
import sys
import json
import time
import random
import hashlib
import platform
import subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader

print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())


## 2. Parámetros, ubicación del proyecto y corpus

La raíz del proyecto debe contener el paquete `minigpt/`. Por defecto se usa **`data/corpus.txt`** desde esa raíz, ya que el corpus no está versionado. Para cambiarlo a DACSA, por ejemplo, poned `ARCHIVO_CORPUS = "dacsa_spanish_articles_train.txt"`.

Ejecutad el notebook desde la raíz del repositorio o desde `experimentos_2_y_3/`. Si el entorno de Jupyter arranca en otra carpeta, definid la variable de entorno `EXP2_PROJECT_ROOT` con la ruta absoluta de vuestro proyecto.

Por defecto solo se lee el comienzo del corpus. Para una comprobación corta sin tocar los parámetros de la memoria, podéis definir `EXP2_SMOKE_TEST=1` **antes** de abrir el kernel; los resultados de ese modo no se deben usar para sacar conclusiones.


In [ ]:
ARCHIVO_CORPUS = 'corpus.txt'  # Cambiadlo por otro archivo de data/ si procede.

CFG = {
    'max_caracteres': 100_000, # No cargar todo el corpus grande en RAM.
    'fraccion_validacion': 0.10,
    'longitud_contexto': 64,
    'stride': 64,              # Ventanas no solapadas dentro de cada partición.
    'd_model': 96,
    'n_capas': 2,
    'n_cabezas': 4,
    'dropout': 0.0,
    'batch_size': 32,
    'epocas': 6,
    'lr': 3e-4,
    'weight_decay': 0.01,
    'max_norm_grad': 1.0,
    'semillas': [42],          # Para comprobar estabilidad: [42, 43, 44].
    'guardar_mejor_modelo': True,
}

if os.environ.get('EXP2_SMOKE_TEST') == '1':
    CFG.update({
        'max_caracteres': 4_000,
        'longitud_contexto': 16,
        'stride': 16,
        'd_model': 32,
        'n_capas': 1,
        'n_cabezas': 4,
        'batch_size': 8,
        'epocas': 2,
        'semillas': [42],
        'guardar_mejor_modelo': False,
    })

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cpu':
    torch.set_num_threads(min(4, max(1, os.cpu_count() or 1)))


def localizar_proyecto():
    # Encontrar primero la implementación REAL del equipo.
    candidatos = []
    if os.environ.get('EXP2_PROJECT_ROOT'):
        candidatos.append(Path(os.environ['EXP2_PROJECT_ROOT']).expanduser().resolve())
    cwd = Path.cwd().resolve()
    candidatos.extend([cwd, *cwd.parents])
    for candidato in candidatos:
        if (candidato / 'minigpt' / 'model.py').is_file() and (candidato / 'minigpt' / 'config.py').is_file():
            return candidato
    raise FileNotFoundError(
        'No encuentro minigpt/model.py y minigpt/config.py. Ejecuta el notebook '
        'desde el repositorio o define EXP2_PROJECT_ROOT con su ruta absoluta.'
    )

RAIZ = localizar_proyecto()
RUTA_CORPUS = RAIZ / 'data' / ARCHIVO_CORPUS
if not RUTA_CORPUS.is_file():
    raise FileNotFoundError(
        f'No se encuentra el corpus {RUTA_CORPUS}. Debéis colocarlo en data/ '
        'o cambiar ARCHIVO_CORPUS por un archivo existente en esa carpeta.'
    )
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

SALIDA = RAIZ / 'experimentos_2_y_3' / 'resultados_exp2'
SALIDA.mkdir(parents=True, exist_ok=True)

assert CFG['d_model'] % CFG['n_cabezas'] == 0
assert (CFG['d_model'] // CFG['n_cabezas']) % 2 == 0, 'RoPE exige dimensión de cabeza par.'
assert 0 < CFG['fraccion_validacion'] < 1
assert CFG['stride'] >= 1
print('Proyecto:', RAIZ)
print('Corpus:', RUTA_CORPUS)
print('Dispositivo:', DEVICE)
print('Resultados:', SALIDA)
print('Config:', CFG)


## 3. Lectura, vocabulario y separación de datos

Se lee un fragmento acotado de `data/corpus.txt`. Primero dividimos el texto en entrenamiento y validación; después creamos ventanas. Así, ninguna ventana de validación contiene caracteres de entrenamiento.

El vocabulario es de **caracteres**, construido solo con el entrenamiento. Su sencillez permite entrenar `GPTModel` sin utilizar los 50.257 tokens BPE de GPT-2, cuyo gran cabezal de salida encarecería este experimento. El `<UNK>` deja explícitos los caracteres de validación desconocidos.

**Observación:** si el corpus está ordenado temáticamente o por idioma, el corte contiguo puede generar distribuciones distintas; comentad esta limitación y mantened fijo el corte al comparar modelos.


In [ ]:
# Lectura limitada: evita cargar por completo un corpus muy grande en memoria.
with RUTA_CORPUS.open('r', encoding='utf-8', errors='replace') as archivo:
    texto = archivo.read(CFG['max_caracteres'])
minimo = 2 * (CFG['longitud_contexto'] + 1)
if len(texto) < minimo:
    raise ValueError(f'El corpus seleccionado necesita al menos {minimo} caracteres para este contexto.')

corte = int(len(texto) * (1 - CFG['fraccion_validacion']))
texto_train = texto[:corte]
texto_val = texto[corte:]
if min(len(texto_train), len(texto_val)) < CFG['longitud_contexto'] + 1:
    raise ValueError('Las particiones deben contener al menos longitud_contexto + 1 caracteres. Aumenta max_caracteres.')

caracteres = sorted(set(texto_train))
stoi = {caracter: i for i, caracter in enumerate(caracteres)}
ID_UNK = len(stoi)
VOCAB_SIZE = ID_UNK + 1

def codificar(cadena):
    return torch.tensor([stoi.get(c, ID_UNK) for c in cadena], dtype=torch.long)

ids_train = codificar(texto_train)
ids_val = codificar(texto_val)
num_desconocidos = int((ids_val == ID_UNK).sum().item())

print(f'Caracteres: train={len(texto_train):,} | validación={len(texto_val):,}')
print(f'Vocabulario (incluido <UNK>): {VOCAB_SIZE}')
print(f'Caracteres desconocidos en validación: {num_desconocidos:,} '
      f'({100 * num_desconocidos / max(1, len(ids_val)):.2f} %)')
print(f'Muestra del corpus: {texto[:180]!r}')

In [ ]:
class VentanasCausales(Dataset):
    '''Una muestra contiene x[t:t+L] y la respuesta x[t+1:t+L+1].'''
    def __init__(self, tokens, longitud_contexto, stride):
        self.tokens = tokens
        self.contexto = longitud_contexto
        self.stride = stride
        if len(tokens) < longitud_contexto + 1:
            raise ValueError('No hay suficientes tokens para formar una ventana causal.')
        self.inicios = range(0, len(tokens) - longitud_contexto, stride)

    def __len__(self):
        return len(self.inicios)

    def __getitem__(self, indice):
        inicio = self.inicios[indice]
        fin = inicio + self.contexto
        return self.tokens[inicio:fin], self.tokens[inicio + 1:fin + 1]

train_ds = VentanasCausales(ids_train, CFG['longitud_contexto'], CFG['stride'])
val_ds = VentanasCausales(ids_val, CFG['longitud_contexto'], CFG['stride'])

def loaders_para_semilla(semilla):
    # Generador NUEVO cada vez: igual permutación para Learned y RoPE.
    generador = torch.Generator().manual_seed(semilla + 10_000)
    train_loader = DataLoader(
        train_ds, batch_size=CFG['batch_size'], shuffle=True,
        generator=generador, num_workers=0, drop_last=False
    )
    val_loader = DataLoader(
        val_ds, batch_size=CFG['batch_size'], shuffle=False, num_workers=0
    )
    return train_loader, val_loader

x_demo, y_demo = train_ds[0]
assert torch.equal(x_demo[1:], y_demo[:-1])
assert x_demo.shape == y_demo.shape == (CFG['longitud_contexto'],)
print(f'Ventanas: entrenamiento={len(train_ds)} | validación={len(val_ds)}')
print(f'Lotes por época: {(len(train_ds) + CFG["batch_size"] - 1) // CFG["batch_size"]}')

## 4. Importar el modelo de vuestra implementación

No redefinimos `MiniGPT`, atención ni RoPE. Las dos variantes se construyen con las clases reales del proyecto:

```python
from minigpt.config import GPTConfig
from minigpt.model import GPTModel

GPTModel(GPTConfig(..., positional="learned"))
GPTModel(GPTConfig(..., positional="rope"))
```

Esto activa la misma arquitectura del paquete: `learned` suma `pos_emb` a los embeddings de token; `rope` rota **Q y K** dentro de `MultiHeadAttention` usando `minigpt/rope.py`. El entrenamiento y los datos son comunes a ambos.


In [ ]:
# Clases importadas directamente de la rama development del equipo.
from minigpt.config import GPTConfig
from minigpt.model import GPTModel
from minigpt.experiments import count_parameters
from minigpt.rope import rope_angles, apply_rotary


def crear_modelo(tipo_posicional):
    config = GPTConfig(
        vocab_size=VOCAB_SIZE,
        context_length=CFG['longitud_contexto'],
        emb_dim=CFG['d_model'],
        n_heads=CFG['n_cabezas'],
        n_layers=CFG['n_capas'],
        drop_rate=CFG['dropout'],
        qkv_bias=True,
        positional=tipo_posicional,
        rope_theta=10_000.0,
    )
    return GPTModel(config)


print('Clase utilizada:', GPTModel.__module__ + '.' + GPTModel.__name__)
print('Código importado desde:', __import__('minigpt.model', fromlist=['__file__']).__file__)


## 5. Comprobaciones **antes** del entrenamiento

1. RoPE debe conservar aproximadamente los productos escalares de atención cuando desplazamos conjuntamente las posiciones de Q y K (prueba numérica sobre **vuestra** función `apply_rotary`).
2. Los dos `GPTModel` deben compartir exactamente todos los tensores comunes de su inicialización aleatoria.
3. Ambos deben producir logits de forma `(batch, contexto, vocabulario)` y admitir retropropagación.
4. La única diferencia de parámetros esperada es la tabla de posiciones `pos_emb` de Learned.

**La misma inicialización aleatoria no significa utilizar pesos preentrenados.**


In [ ]:
def fijar_semilla(semilla):
    random.seed(semilla)
    np.random.seed(semilla)
    torch.manual_seed(semilla)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(semilla)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def crear_par_comparable(semilla):
    # model.py construye pos_emb al final para preservar el mismo RNG inicial
    # en todos los pesos que comparten Learned y RoPE.
    fijar_semilla(semilla)
    learned = crear_modelo('learned')
    fijar_semilla(semilla)
    rope = crear_modelo('rope')
    estado_learned = learned.state_dict()
    estado_rope = rope.state_dict()
    claves_comunes = sorted(set(estado_learned) & set(estado_rope))
    assert claves_comunes, 'No hay pesos iniciales comparables.'
    for clave in claves_comunes:
        torch.testing.assert_close(estado_learned[clave], estado_rope[clave], rtol=0, atol=0)
    assert 'pos_emb.weight' in estado_learned and 'pos_emb.weight' not in estado_rope
    return learned, rope


fijar_semilla(42)
dim_cabeza = CFG['d_model'] // CFG['n_cabezas']
q = torch.randn(2, CFG['n_cabezas'], 7, dim_cabeza)
k = torch.randn_like(q)
cos0, sin0 = rope_angles(7, dim_cabeza, offset=0)
cos1, sin1 = rope_angles(7, dim_cabeza, offset=11)
scores0 = apply_rotary(q, cos0, sin0) @ apply_rotary(k, cos0, sin0).transpose(-2, -1)
scores1 = apply_rotary(q, cos1, sin1) @ apply_rotary(k, cos1, sin1).transpose(-2, -1)
torch.testing.assert_close(scores0, scores1, rtol=1e-4, atol=1e-4)

learned_test, rope_test = crear_par_comparable(42)
x_test, y_test = next(iter(DataLoader(train_ds, batch_size=2)))
for modelo in (learned_test, rope_test):
    logits = modelo(x_test)
    loss = F.cross_entropy(logits.reshape(-1, VOCAB_SIZE), y_test.reshape(-1))
    assert logits.shape == (2, CFG['longitud_contexto'], VOCAB_SIZE)
    assert torch.isfinite(loss).item()
    loss.backward()
    assert modelo.tok_emb.weight.grad is not None
    assert torch.isfinite(modelo.tok_emb.weight.grad).all().item()

n_learned = count_parameters(learned_test)
n_rope = count_parameters(rope_test)
assert n_learned - n_rope == CFG['longitud_contexto'] * CFG['d_model']
print('Desplazamiento relativo RoPE: OK')
print('Dimensiones, loss y backward: OK')
print('Todos los pesos y buffers comunes inicializados de forma idéntica: OK')
print(f'Parámetros: Learned={n_learned:,} | RoPE={n_rope:,} | diferencia={n_learned - n_rope:,}')
del learned_test, rope_test


## 6. Entrenar desde cero **los modelos reales**

Se realiza predicción del siguiente carácter con logits producidos por `GPTModel(x)` y la función `torch.nn.functional.cross_entropy`. No se cargan pesos de GPT-2 ni se hace fine-tuning.

- **Un recorrido completo por las ventanas = una época.** Learned y RoPE ven las mismas ventanas en el mismo orden, con el mismo optimizador y la misma semilla.
- Se reporta la `train_loss` media observada durante los pasos de la época, y `val_loss` con los pesos congelados y `eval()` al inicio y al final de cada época.
- Con `CFG['guardar_mejor_modelo']=True`, se guardan los mejores `state_dict` según validación en `experimentos_2_y_3/pesos/{learned,rope}/best_exp2_seed...pt`. No se guardan los textos del corpus.

Para publicar resultados más robustos, repetid luego con las semillas `[42, 43, 44]` y comparad la variabilidad; la prueba predeterminada usa solo una.


In [ ]:
@torch.no_grad()
def evaluar(modelo, cargador):
    previo = modelo.training
    modelo.eval()
    suma_loss = 0.0
    total_tokens = 0
    for entradas, objetivos in cargador:
        entradas = entradas.to(DEVICE)
        objetivos = objetivos.to(DEVICE)
        logits = modelo(entradas)
        # Se pondera por número de tokens, no por número de batches.
        suma_loss += F.cross_entropy(
            logits.reshape(-1, logits.size(-1)).float(),
            objetivos.reshape(-1), reduction='sum'
        ).item()
        total_tokens += objetivos.numel()
    if previo:
        modelo.train()
    return suma_loss / total_tokens


def guardar_checkpoint(modelo, etiqueta, semilla, epoca, val_loss):
    carpeta = RAIZ / 'experimentos_2_y_3' / 'pesos' / etiqueta.lower()
    carpeta.mkdir(parents=True, exist_ok=True)
    destino = carpeta / f'best_exp2_seed{semilla}.pt'
    # Copias de parámetros hacia CPU: checkpoint portable incluso si se entrenó con CUDA.
    estado_cpu = {k: v.detach().cpu().clone() for k, v in modelo.state_dict().items()}
    torch.save({
        'model_state_dict': estado_cpu,
        'model_config': vars(modelo.cfg).copy(),
        'semilla': semilla,
        'epoca': epoca,
        'val_loss': val_loss,
    }, destino)
    return destino


def entrenar(modelo, etiqueta, semilla):
    modelo = modelo.to(DEVICE)
    optimizador = torch.optim.AdamW(
        modelo.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay']
    )
    cargador_train, cargador_val = loaders_para_semilla(semilla)
    historial_epocas, historial_pasos = [], []
    val_inicial = evaluar(modelo, cargador_val)
    historial_epocas.append({
        'semilla': semilla, 'modelo': etiqueta, 'epoca': 0,
        'train_loss': np.nan, 'val_loss': val_inicial, 'segundos_epoca': 0.0
    })
    mejor_val = float('inf')
    contador_pasos = 0

    for epoca in range(1, CFG['epocas'] + 1):
        modelo.train()
        suma_loss, total_tokens = 0.0, 0
        inicio_epoca = time.perf_counter()
        for entradas, objetivos in cargador_train:
            entradas = entradas.to(DEVICE)
            objetivos = objetivos.to(DEVICE)
            optimizador.zero_grad(set_to_none=True)
            logits = modelo(entradas)
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)).float(), objetivos.reshape(-1)
            )
            if not torch.isfinite(loss).item():
                raise ValueError(f'Loss no finita: {etiqueta}, época {epoca}.')
            loss.backward()
            torch.nn.utils.clip_grad_norm_(modelo.parameters(), CFG['max_norm_grad'])
            optimizador.step()
            contador_pasos += 1
            suma_loss += loss.item() * objetivos.numel()
            total_tokens += objetivos.numel()
            historial_pasos.append({
                'semilla': semilla, 'modelo': etiqueta, 'epoca': epoca,
                'paso': contador_pasos, 'train_loss_paso': loss.item()
            })

        segundos = time.perf_counter() - inicio_epoca
        perdida_train = suma_loss / total_tokens
        perdida_val = evaluar(modelo, cargador_val)
        if perdida_val < mejor_val:
            mejor_val = perdida_val
            if CFG['guardar_mejor_modelo']:
                guardar_checkpoint(modelo, etiqueta, semilla, epoca, perdida_val)
        historial_epocas.append({
            'semilla': semilla, 'modelo': etiqueta, 'epoca': epoca,
            'train_loss': perdida_train, 'val_loss': perdida_val,
            'segundos_epoca': segundos
        })
        print(f'{etiqueta:7s} | semilla={semilla} | época={epoca}/{CFG["epocas"]} '
              f'| train={perdida_train:.4f} | val={perdida_val:.4f} | {segundos:.1f}s')

    del optimizador
    return historial_epocas, historial_pasos


In [ ]:
# Esta es la celda de entrenamiento principal. Se ejecuta únicamente al decidir correr el experimento.
registro_epocas = []
registro_pasos = []

for semilla in CFG['semillas']:
    learned, rope = crear_par_comparable(semilla)
    print(f'\n=== Semilla {semilla}: Learned ===')
    ep, pasos = entrenar(learned, 'Learned', semilla)
    registro_epocas.extend(ep)
    registro_pasos.extend(pasos)
    del learned
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

    print(f'\n=== Semilla {semilla}: RoPE ===')
    ep, pasos = entrenar(rope, 'RoPE', semilla)
    registro_epocas.extend(ep)
    registro_pasos.extend(pasos)
    del rope
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

df_epocas = pd.DataFrame(registro_epocas)
df_pasos = pd.DataFrame(registro_pasos)
assert len(df_epocas) == len(CFG['semillas']) * 2 * (CFG['epocas'] + 1)
assert len(df_pasos.groupby(['semilla', 'modelo'])) == len(CFG['semillas']) * 2
assert df_epocas['val_loss'].notna().all()
print('Entrenamiento completado.')
display(df_epocas.round(4))

## 7. Curvas de pérdida

Se muestran las pérdidas con **los mismos ejes** para Learned y RoPE. Con una semilla aparece una línea por modelo; con varias, la línea representa la media y la banda representa ±1 desviación típica. En las curvas por paso se suaviza mediante media móvil **solo para visualización**; el CSV conserva cada pérdida original.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.8))
for etiqueta, color in [('Learned', 'tab:blue'), ('RoPE', 'tab:orange')]:
    parte = df_epocas[df_epocas['modelo'] == etiqueta]
    media = parte.groupby('epoca')['val_loss'].mean()
    desv = parte.groupby('epoca')['val_loss'].std().fillna(0)
    ax.plot(media.index, media.values, marker='o', label=etiqueta, color=color)
    if len(CFG['semillas']) > 1:
        ax.fill_between(media.index, media.values - desv.values, media.values + desv.values,
                        color=color, alpha=0.15)
ax.set(title='Convergencia en validación: Learned vs. RoPE',
       xlabel='Época (0 = antes de entrenar)', ylabel='Loss de validación (cross-entropy)')
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(SALIDA / 'exp2_val_loss_epocas.png', dpi=200)
plt.show()

fig, ax = plt.subplots(figsize=(8, 4.8))
for etiqueta, color in [('Learned', 'tab:blue'), ('RoPE', 'tab:orange')]:
    parte = df_epocas[(df_epocas['modelo'] == etiqueta) & (df_epocas['epoca'] > 0)]
    media = parte.groupby('epoca')['train_loss'].mean()
    desv = parte.groupby('epoca')['train_loss'].std().fillna(0)
    ax.plot(media.index, media.values, marker='o', label=etiqueta, color=color)
    if len(CFG['semillas']) > 1:
        ax.fill_between(media.index, media.values - desv.values, media.values + desv.values,
                        color=color, alpha=0.15)
ax.set(title='Pérdida media de entrenamiento por época',
       xlabel='Época', ylabel='Loss de entrenamiento (cross-entropy)')
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(SALIDA / 'exp2_train_loss_epocas.png', dpi=200)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.8))
VENTANA_MEDIA_MOVIL = 10
for etiqueta, color in [('Learned', 'tab:blue'), ('RoPE', 'tab:orange')]:
    parte = df_pasos[df_pasos['modelo'] == etiqueta].sort_values(['semilla', 'paso'])
    # Si hay varias semillas, se calcula primero la media por número de paso.
    serie = parte.groupby('paso')['train_loss_paso'].mean().sort_index()
    suavizada = serie.rolling(VENTANA_MEDIA_MOVIL, min_periods=1).mean()
    ax.plot(suavizada.index, suavizada.values, label=etiqueta, color=color)
ax.set(title=f'Pérdida de entrenamiento por actualización (media móvil de {VENTANA_MEDIA_MOVIL})',
       xlabel='Paso de optimización', ylabel='Loss de entrenamiento')
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(SALIDA / 'exp2_train_loss_pasos.png', dpi=200)
plt.show()

## 8. Tabla comparativa y análisis

Una menor *loss* de validación en épocas tempranas es evidencia a favor de una mejor convergencia **en esta configuración**. También debe comprobarse si ambas variantes aprenden y si existe una diferencia estable entre semillas; no basta con elegir la curva más bonita.

La columna `AUC_val` aproxima el área bajo la curva de validación entre las épocas 0 y N mediante la regla del trapecio. **Menor AUC** indica menor pérdida acumulada a lo largo del entrenamiento, con idéntico número de épocas. No expresa superioridad estadística por sí sola.

In [ ]:
filas_resumen = []
for (semilla, etiqueta), parte in df_epocas.groupby(['semilla', 'modelo']):
    parte = parte.sort_values('epoca')
    losses = parte['val_loss'].to_numpy()
    epocas = parte['epoca'].to_numpy()
    auc = float(np.sum((losses[:-1] + losses[1:]) / 2 * np.diff(epocas)))
    filas_resumen.append({
        'semilla': semilla,
        'modelo': etiqueta,
        'val_inicial': losses[0],
        'val_epoca_1': losses[1],
        'val_epoca_2': losses[2] if len(losses) > 2 else np.nan,
        'val_final': losses[-1],
        'reduccion_val': losses[0] - losses[-1],
        'AUC_val': auc,
    })

resumen = pd.DataFrame(filas_resumen)
resumen_media = resumen.groupby('modelo').agg(
    val_inicial=('val_inicial', 'mean'),
    val_epoca_1=('val_epoca_1', 'mean'),
    val_epoca_2=('val_epoca_2', 'mean'),
    val_final=('val_final', 'mean'),
    reduccion_val=('reduccion_val', 'mean'),
    AUC_val=('AUC_val', 'mean'),
)
print('Resumen por semilla:')
display(resumen.round(4))
print('Media por arquitectura:')
display(resumen_media.round(4))

if len(CFG['semillas']) == 1:
    print('ADVERTENCIA: un único par de entrenamientos es un estudio exploratorio; '
          'para valorar estabilidad ejecutar al menos tres semillas.')

auc_rope = resumen_media.loc['RoPE', 'AUC_val']
auc_learned = resumen_media.loc['Learned', 'AUC_val']
dif_auc = auc_rope - auc_learned
if dif_auc < 0:
    print(f'Observación: RoPE presenta menor AUC de validación ({abs(dif_auc):.4f} puntos).')
elif dif_auc > 0:
    print(f'Observación: Learned presenta menor AUC de validación ({abs(dif_auc):.4f} puntos).')
else:
    print('Observación: ambas AUC de validación coinciden numéricamente.')
print('Esto NO prueba una diferencia estadísticamente significativa ni una ventaja universal.')

## 9. Exportación y reproducibilidad

Los archivos se guardan en `experimentos_2_y_3/resultados_exp2/`:

- `exp2_loss_epocas.csv`: entrenamiento y validación en cada época, incluido el estado inicial (época 0).
- `exp2_loss_pasos.csv`: entrenamiento en cada actualización del optimizador.
- `exp2_resumen_semillas.csv` y `exp2_resumen_media.csv`: comparación entre variantes.
- `exp2_val_loss_epocas.png`, `exp2_train_loss_epocas.png`, `exp2_train_loss_pasos.png`.
- `exp2_configuracion.json`: arquitectura, dispositivo, semilla, corpus utilizado y hash del fragmento.

Con la opción correspondiente se guardan además checkpoints en la carpeta `pesos/` ya existente en el repositorio. No hace falta subir a GitHub el corpus ni los pesos.


In [ ]:
def git_commit(carpeta):
    try:
        return subprocess.check_output(
            ['git', '-C', str(carpeta), 'rev-parse', 'HEAD'],
            text=True, stderr=subprocess.DEVNULL, timeout=5
        ).strip()
    except (OSError, subprocess.CalledProcessError, subprocess.TimeoutExpired):
        return None

metadatos = {
    'experimento': 'Exp. 2 - Convergencia desde cero (GPTModel Learned vs RoPE)',
    'codigo_modelo': 'minigpt.model.GPTModel',
    'codigo_configuracion': 'minigpt.config.GPTConfig',
    'codigo_rope': 'minigpt.rope.apply_rotary',
    'configuracion': CFG,
    'tokenizador': 'caracteres; vocabulario de entrenamiento y UNK para validacion',
    'archivo_corpus': f'data/{ARCHIVO_CORPUS}',
    'caracteres_utilizados': len(texto),
    'sha256_texto_utilizado': hashlib.sha256(texto.encode('utf-8')).hexdigest(),
    'vocab_size': VOCAB_SIZE,
    'n_ventanas_train': len(train_ds),
    'n_ventanas_val': len(val_ds),
    'n_caracteres_unk_val': num_desconocidos,
    'parametros_learned': n_learned,
    'parametros_rope': n_rope,
    'dispositivo': str(DEVICE),
    'pytorch': torch.__version__,
    'python': platform.python_version(),
    'git_commit': git_commit(RAIZ),
}

df_epocas.to_csv(SALIDA / 'exp2_loss_epocas.csv', index=False)
df_pasos.to_csv(SALIDA / 'exp2_loss_pasos.csv', index=False)
resumen.to_csv(SALIDA / 'exp2_resumen_semillas.csv', index=False)
resumen_media.to_csv(SALIDA / 'exp2_resumen_media.csv')
with (SALIDA / 'exp2_configuracion.json').open('w', encoding='utf-8') as f:
    json.dump(metadatos, f, ensure_ascii=False, indent=2, default=str)

print('Resultados guardados:')
for ruta in sorted(SALIDA.glob('exp2_*')):
    print(' -', ruta.name)


## 10. Cómo interpretar los resultados y trasladarlos a la memoria

**Qué se debe verificar:**

1. Las curvas `train_loss` deben mostrar aprendizaje; comprobad por separado la evolución de `val_loss`.
2. Comparad pérdidas de validación en épocas **1 y 2**, así como la reducción respecto de la **época 0**, porque una menor pérdida inicial no implica converger más deprisa.
3. La **AUC** de validación (regla trapezoidal entre épocas 0 y N) resume la pérdida acumulada: cuanto menor, mejor en las mismas condiciones.
4. Si las curvas se cruzan o la dirección de la diferencia cambia con la semilla, indicadlo. RoPE **no garantiza** ser superior en aprendizaje temprano.
5. La prueba numérica de invariancia de RoPE valida una propiedad de la implementación, pero no prueba una mayor velocidad de convergencia.

**Texto orientativo para completar con resultados reales:**

> Utilizamos la implementación de `GPTModel` del proyecto con dos configuraciones posicionales (`learned` y `rope`). Se entrenaron desde cero modelos de [N] capas, [D] dimensiones y [H] cabezas sobre [N] caracteres de [corpus], con igual partición, parámetros de optimización, inicialización compartida y orden de lotes. Tras [E] épocas, Learned obtuvo [X] de pérdida de validación y RoPE [Y]. Las pérdidas correspondientes a las primeras épocas fueron [valores]. La comparación [apoya / no apoya] una convergencia más rápida de RoPE **en esta configuración**, teniendo en cuenta [variabilidad entre semillas y demás limitaciones].

**Limitaciones:** entrenamiento reducido, tokenización por caracteres en vez de BPE de GPT-2, solo un subconjunto inicial de corpus, posible diferencia entre segmentos contiguos, distinto número de parámetros de posición y sensibilidad a las semillas. No extrapolar a GPT-2 de 124 M ni a modelos mayores.

**Referencias:** Raschka (2024), *Build a Large Language Model (From Scratch)*; Su et al. (2021), *RoFormer: Enhanced Transformer with Rotary Position Embedding*.
